# TFM — Matching semántico de conceptos técnicos en presupuestos históricos

Este notebook contiene la fase experimental del TFM.

## Objetivo
Comparar distintos enfoques de recuperación de conceptos: baseline léxico (TF-IDF), embeddings semánticos, modelo híbrido (embeddings + medida) y reranking con cross-encoders.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder


## 1. Carga de datos

Se cargan dos ficheros CSV:

- `catalogo_historicos_starter.csv`: catálogo de conceptos históricos.
- `ejemplos_queries_starter.csv`: consultas nuevas con su concepto histórico esperado etiquetado.

In [2]:
base_path = Path("..")
data_path = base_path / "data"

historicos_path = data_path / "catalogo_historicos_starter.csv"
queries_path = data_path / "ejemplos_queries_starter.csv"

historicos = pd.read_csv(historicos_path)
queries = pd.read_csv(queries_path)

print("Históricos:", historicos.shape)
print("Queries:", queries.shape)


Históricos: (1268, 7)
Queries: (42, 6)


### Exploración rápida


In [3]:
print("Columnas de históricos:")
print(historicos.columns.tolist())

print("\nColumnas de queries:")
print(queries.columns.tolist())

display(historicos.head(5))
display(queries.head(5))


Columnas de históricos:
['sheet', 'code', 'concepto_historico', 'unidad', 'precio_base', 'med_int', 'med_ext']

Columnas de queries:
['concepto_query', 'concepto_historico_esperado', 'sheet_origen', 'code', 'unidad', 'precio_base']


,sheet,code,concepto_historico,unidad,precio_base,med_int,med_ext
0,DMT,PIJ613,"TOR P/TABLAROCA 6x1/2"""" ST",PZA,0.05,NaN,NaN
1,DMT,PIJ619,"TOR P/TABLAROCA 6x3/4"""" ST",PZA,0.15,NaN,NaN
2,DMT,PIJ625,"TOR P/TABLAROCA 6x1"""" ST",PZA,0.11,NaN,NaN
3,DMT,PIJ623,"TOR P/TABLAROCA 6x1 1/8"""" ST",PZA,0.10,NaN,NaN
4,DMT,PIJ625,"TOR P/TABLAROCA 6x1 1/2"""" ST",PZA,0.12,NaN,NaN


,concepto_query,concepto_historico_esperado,sheet_origen,code,unidad,precio_base
0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,DMT,ARP12X412,PZA,9.26
1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,DMT,PIJGALV8112,PZA,0.25
2,TAQUETE ARPON 1/4X2 1/4,TAQUETE ARPON 1/4X2 1/4,DMT,ARP14X214,PZA,2.04
3,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,DMT,BBPCP1X12,PZA,122.81
4,"TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV","TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV",DMT,PIJTRUSSG8X112,PZA,0.23


## 2. Preprocesado de texto

Se normaliza el texto para reducir variaciones superficiales que no deberían afectar al matching:
- paso a minúsculas,
- eliminación de tildes,
- limpieza de caracteres especiales y símbolos técnicos.

In [4]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    
    # pasar a string y minúsculas
    texto = str(texto).lower()
    
    # quitar tildes
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    
    # reemplazar algunos símbolos comunes
    texto = texto.replace("ø", " diametro ")
    texto = texto.replace('"', " pulgadas ")
    texto = texto.replace("'", " ")
    texto = texto.replace("/", " ")
    texto = texto.replace("-", " ")
    texto = texto.replace("(", " ")
    texto = texto.replace(")", " ")
    texto = texto.replace(",", " ")
    texto = texto.replace(".", " ")
    
    # quitar todo lo que no sea texto/número/espacio
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    
    # quitar espacios repetidos
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto


In [5]:
historicos["concepto_limpio"] = historicos["concepto_historico"].apply(limpiar_texto)
queries["query_limpia"] = queries["concepto_query"].apply(limpiar_texto)
queries["esperado_limpio"] = queries["concepto_historico_esperado"].apply(limpiar_texto)


In [6]:
print("Conceptos históricos únicos (original):", historicos["concepto_historico"].nunique())
print("Conceptos históricos únicos (limpios):", historicos["concepto_limpio"].nunique())


Conceptos históricos únicos (original): 1213
Conceptos históricos únicos (limpios): 1213


## 3. Método 1 — Baseline léxico con TF-IDF

Este baseline se incluye como referencia clásica.  
Su función es medir hasta qué punto un enfoque puramente léxico es suficiente antes de pasar a modelos semánticos.

In [7]:
vectorizer = TfidfVectorizer()

X_historicos = vectorizer.fit_transform(historicos["concepto_limpio"])
X_queries = vectorizer.transform(queries["query_limpia"])

sim_matrix_tfidf = cosine_similarity(X_queries, X_historicos)

best_idx_tfidf = sim_matrix_tfidf.argmax(axis=1)
best_scores_tfidf = sim_matrix_tfidf.max(axis=1)

resultados_baseline = queries.copy()
resultados_baseline["match_idx"] = best_idx_tfidf
resultados_baseline["score_top1_tfidf"] = best_scores_tfidf
resultados_baseline["concepto_match_top1_tfidf"] = historicos.iloc[best_idx_tfidf]["concepto_historico"].values

resultados_baseline["acierto_top1_tfidf"] = (
    resultados_baseline["concepto_match_top1_tfidf"].str.strip().str.lower()
    ==
    resultados_baseline["concepto_historico_esperado"].str.strip().str.lower()
)

accuracy_tfidf = resultados_baseline["acierto_top1_tfidf"].mean()

print("TF-IDF Top-1 accuracy:", round(accuracy_tfidf, 6))
print("Aciertos:", resultados_baseline["acierto_top1_tfidf"].sum(), "de", len(resultados_baseline))


TF-IDF Top-1 accuracy: 0.214286
Aciertos: 9 de 42


## 4. Comparativa de modelos de embeddings

Se comparan varios modelos de embeddings para identificar cuál ofrece mejor recuperación inicial.  
Aquí el objetivo no es todavía el reranking final, sino seleccionar el mejor candidato para las etapas siguientes.

In [8]:
def evaluar_embedding_model(model, model_name, historicos, queries):
    emb_historicos = model.encode(
        historicos["concepto_limpio"].tolist(),
        convert_to_numpy=True,
        show_progress_bar=True
    )

    emb_queries = model.encode(
        queries["query_limpia"].tolist(),
        convert_to_numpy=True,
        show_progress_bar=True
    )

    sim_matrix = cosine_similarity(emb_queries, emb_historicos)

    best_idx = sim_matrix.argmax(axis=1)
    best_scores = sim_matrix.max(axis=1)

    resultados = queries.copy()
    resultados["match_idx"] = best_idx
    resultados["score_top1_emb"] = best_scores
    resultados["concepto_match_top1_emb"] = historicos.iloc[best_idx]["concepto_historico"].values
    resultados["sheet_match_emb"] = historicos.iloc[best_idx]["sheet"].values if "sheet" in historicos.columns else None
    resultados["code_match_emb"] = historicos.iloc[best_idx]["code"].values if "code" in historicos.columns else None

    resultados["acierto_top1_exacto_emb"] = (
        resultados["concepto_match_top1_emb"].str.strip().str.lower()
        ==
        resultados["concepto_historico_esperado"].str.strip().str.lower()
    )

    accuracy_top1 = resultados["acierto_top1_exacto_emb"].mean()

    return {
        "model_name": model_name,
        "emb_historicos": emb_historicos,
        "emb_queries": emb_queries,
        "sim_matrix": sim_matrix,
        "resultados": resultados,
        "accuracy_top1": accuracy_top1
    }


def calcular_metricas_ranking(sim_matrix, historicos, queries, top_ks=(1, 3, 5)):
    resultados = {}
    top_indices = np.argsort(-sim_matrix, axis=1)

    expected = queries["concepto_historico_esperado"].str.strip().str.lower().tolist()
    candidatos = historicos["concepto_historico"].str.strip().str.lower().tolist()

    for k in top_ks:
        aciertos = 0
        for i in range(len(queries)):
            idxs = top_indices[i, :k]
            candidatos_topk = [candidatos[j] for j in idxs]
            if expected[i] in candidatos_topk:
                aciertos += 1
        resultados[f"top{k}_accuracy"] = aciertos / len(queries)

    reciprocal_ranks = []
    for i in range(len(queries)):
        idxs = top_indices[i]
        rank_encontrado = None
        for rank, idx in enumerate(idxs, start=1):
            if candidatos[idx] == expected[i]:
                rank_encontrado = rank
                break
        reciprocal_ranks.append(1 / rank_encontrado if rank_encontrado is not None else 0)

    resultados["mrr"] = np.mean(reciprocal_ranks)
    return resultados


In [9]:
embedding_models = {
    "MiniLM_multilingual": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "multilingual_e5_base": "intfloat/multilingual-e5-base",
    "bge_m3": "BAAI/bge-m3"
}

resultados_modelos = {}
resumen_embeddings = []

for nombre_modelo, model_id in embedding_models.items():
    print(f"\n=== Evaluando {nombre_modelo} ===")
    model = SentenceTransformer(model_id)

    salida = evaluar_embedding_model(model, nombre_modelo, historicos, queries)
    metricas_ranking = calcular_metricas_ranking(
        salida["sim_matrix"], historicos, queries, top_ks=(1, 3, 5)
    )

    resultados_modelos[nombre_modelo] = salida
    resumen_embeddings.append({
        "modelo": nombre_modelo,
        "model_id": model_id,
        "top1_accuracy": salida["accuracy_top1"],
        "top3_accuracy": metricas_ranking["top3_accuracy"],
        "top5_accuracy": metricas_ranking["top5_accuracy"],
        "mrr": metricas_ranking["mrr"]
    })

resumen_embeddings_df = pd.DataFrame(resumen_embeddings).sort_values(
    by="top1_accuracy", ascending=False
)

display(resumen_embeddings_df)



=== Evaluando MiniLM_multilingual ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...


=== Evaluando bge_m3 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

,modelo,model_id,top1_accuracy,top3_accuracy,top5_accuracy,mrr
1,multilingual_e5_base,intfloat/multilingual-e5-base,0.880952,0.904762,0.904762,0.889304
2,bge_m3,BAAI/bge-m3,0.857143,0.904762,0.904762,0.882932
0,MiniLM_multilingual,sentence-transformers/paraphrase-multilingual-...,0.714286,0.809524,0.857143,0.770698


In [10]:
mejor_modelo_nombre = resumen_embeddings_df.iloc[0]["modelo"]
mejor_resultado = resultados_modelos[mejor_modelo_nombre]
sim_matrix_best = mejor_resultado["sim_matrix"]
resultados_emb_best = mejor_resultado["resultados"].copy()

print("Mejor modelo de embeddings:", mejor_modelo_nombre)
print("Top-1 accuracy:", round(mejor_resultado["accuracy_top1"], 6))


Mejor modelo de embeddings: multilingual_e5_base
Top-1 accuracy: 0.880952


## 5. Modelo híbrido: embeddings + coincidencia de medida

El embedding se usa para recuperar los mejores candidatos y, sobre ese subconjunto, se introduce una señal de dominio:
la coincidencia de medida técnica (pulgadas, fracciones, dimensiones compuestas).  
Si hay candidatos con medida coincidente, se prioriza el mejor entre ellos; si no, se llama al top-1 del embedding.

In [11]:
def extraer_medida_v2(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).lower()

    # Se eliminan patrones técnicos que introducen números no interpretables como medida principal
    texto = re.sub(r'ced[\s\-]?80', ' ', texto)
    texto = re.sub(r'c[\s\-]?80', ' ', texto)
    texto = re.sub(r'dwv', ' ', texto)
    texto = re.sub(r'cts', ' ', texto)
    texto = re.sub(r'rd[\-\s]?\d+(\.\d+)?', ' ', texto)

    # Se buscan medidas compuestas, fracciones y medidas numéricas simples
    patrones = re.findall(r'\d+x\d+\s\d+/\d+|\d+x\d+/\d+|\d+\s\d+/\d+|\d+/\d+|\d+(?:\.\d+)?', texto)
    return " | ".join(patrones) if patrones else None


In [12]:
top_k_emb = 5
top_indices_emb = np.argsort(-sim_matrix_best, axis=1)[:, :top_k_emb]
top_scores_emb = np.take_along_axis(sim_matrix_best, top_indices_emb, axis=1)

top5_rows = []

for i, query_row in queries.iterrows():
    medida_q = extraer_medida_v2(query_row["concepto_query"])

    for rank in range(top_k_emb):
        idx_hist = top_indices_emb[i, rank]
        hist_row = historicos.iloc[idx_hist]
        medida_h = extraer_medida_v2(hist_row["concepto_historico"])

        top5_rows.append({
            "query_idx": i,
            "concepto_query": query_row["concepto_query"],
            "concepto_esperado": query_row["concepto_historico_esperado"],
            "rank_emb": rank + 1,
            "concepto_candidato": hist_row["concepto_historico"],
            "score_emb": top_scores_emb[i, rank],
            "medida_query": medida_q,
            "medida_candidato": medida_h,
            "medida_coincide": (str(medida_q) == str(medida_h)),
            "sheet_candidato": hist_row["sheet"] if "sheet" in historicos.columns else None,
            "code_candidato": hist_row["code"] if "code" in historicos.columns else None
        })

top5_emb_df = pd.DataFrame(top5_rows)
display(top5_emb_df.head(10))


,query_idx,concepto_query,concepto_esperado,rank_emb,concepto_candidato,score_emb,medida_query,medida_candidato,medida_coincide,sheet_candidato,code_candidato
0,0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,1,TAQUETE ARPON 1/2X4 1/2,1.000000,1/2 | 4 1/2,1/2 | 4 1/2,True,DMT,ARP12X412
1,0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,2,TAQUETE ARPON 1/2X4 1/4,0.997746,1/2 | 4 1/2,1/2 | 4 1/4,False,DMT,ARP12X414
2,0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,3,TAQUETE ARPON 1/4X2 1/4,0.995289,1/2 | 4 1/2,1/4 | 2 1/4,False,DMT,ARP14X214
3,0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,4,TAQUETE ARPON 1/2X2 3/4,0.991828,1/2 | 4 1/2,1/2 | 2 3/4,False,DMT,ARP12X234
4,0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,5,TAQUETE ARPON 1/4X1 3/4,0.983228,1/2 | 4 1/2,1/4 | 1 3/4,False,DMT,ARP14X134
5,1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,1,PIJA C/FIJ GALV 8X1 1/2,1.000000,8x1 1/2,8x1 1/2,True,DMT,PIJGALV8112
6,1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,2,PIJA C/FIJ GALV 8X2 1/2,0.992213,8x1 1/2,8x2 1/2,False,DMT,PIJGALV8212
7,1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,3,PIJA C/FIJ GALV 8X1 1/4,0.989606,8x1 1/2,8x1 1/4,False,DMT,PIJGALV8114
8,1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,4,PIJA C/FIJ GALV 8X1,0.986519,8x1 1/2,8 | 1,False,DMT,PIJGALV81
9,1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,5,PIJA C/FIJ GALV 8X2,0.978058,8x1 1/2,8 | 2,False,DMT,PIJGALV82


In [13]:
selecciones_hibridas = []

for query_idx, grupo in top5_emb_df.groupby("query_idx"):
    grupo = grupo.sort_values("rank_emb").copy()
    compatibles = grupo[grupo["medida_coincide"] == True]

    if len(compatibles) > 0:
        elegido = compatibles.iloc[0]
        criterio = "top_emb_con_medida"
    else:
        elegido = grupo.iloc[0]
        criterio = "top_emb_sin_medida"

    selecciones_hibridas.append({
        "query_idx": query_idx,
        "concepto_query": elegido["concepto_query"],
        "concepto_esperado": elegido["concepto_esperado"],
        "concepto_match_hibrido": elegido["concepto_candidato"],
        "score_emb_hibrido": elegido["score_emb"],
        "medida_query": elegido["medida_query"],
        "medida_match": elegido["medida_candidato"],
        "medida_coincide": elegido["medida_coincide"],
        "criterio_seleccion": criterio
    })

resultados_hibrido = pd.DataFrame(selecciones_hibridas)
resultados_hibrido["acierto_top1_hibrido"] = (
    resultados_hibrido["concepto_match_hibrido"].str.strip().str.lower()
    ==
    resultados_hibrido["concepto_esperado"].str.strip().str.lower()
)

accuracy_hibrido = resultados_hibrido["acierto_top1_hibrido"].mean()

print("Híbrido (embedding + medida) Top-1 accuracy:", round(accuracy_hibrido, 6))
print("Aciertos:", resultados_hibrido["acierto_top1_hibrido"].sum(), "de", len(resultados_hibrido))


Híbrido (embedding + medida) Top-1 accuracy: 0.880952
Aciertos: 37 de 42


## 6. Reranking con cross-encoders

Se evalúan varios cross-encoders sobre el Top-5 recuperado por el mejor embedding.
Después, se comprueba si la incorporación de la coincidencia de medida mejora también en este nivel.

In [14]:
def rerank_con_cross_encoder(topk_df, cross_encoder):
    filas_rerankeadas = []

    for query_idx, grupo in topk_df.groupby("query_idx"):
        grupo = grupo.copy().sort_values("rank_emb")
        query_text = grupo.iloc[0]["concepto_query"]

        pairs = [[query_text, candidato] for candidato in grupo["concepto_candidato"].tolist()]
        ce_scores = cross_encoder.predict(pairs)

        grupo["score_cross"] = ce_scores
        grupo = grupo.sort_values("score_cross", ascending=False).copy()
        grupo["rank_cross"] = range(1, len(grupo) + 1)

        filas_rerankeadas.append(grupo)

    return pd.concat(filas_rerankeadas, ignore_index=True)


def evaluar_cross_encoder(topk_df, cross_encoder):
    topk_cross_df = rerank_con_cross_encoder(topk_df, cross_encoder)

    selecciones_cross = []
    for query_idx, grupo in topk_cross_df.groupby("query_idx"):
        elegido = grupo.sort_values("rank_cross").iloc[0]
        selecciones_cross.append({
            "query_idx": query_idx,
            "concepto_query": elegido["concepto_query"],
            "concepto_esperado": elegido["concepto_esperado"],
            "concepto_match_cross": elegido["concepto_candidato"],
            "score_cross": elegido["score_cross"],
            "score_emb_original": elegido["score_emb"],
            "rank_emb_original": elegido["rank_emb"],
            "medida_query": elegido["medida_query"],
            "medida_candidato": elegido["medida_candidato"],
            "medida_coincide": elegido["medida_coincide"]
        })

    resultados_cross = pd.DataFrame(selecciones_cross)
    resultados_cross["acierto_top1_cross"] = (
        resultados_cross["concepto_match_cross"].str.strip().str.lower()
        ==
        resultados_cross["concepto_esperado"].str.strip().str.lower()
    )
    accuracy_cross = resultados_cross["acierto_top1_cross"].mean()

    return topk_cross_df, resultados_cross, accuracy_cross


def evaluar_cross_encoder_con_medida(topk_cross_df):
    selecciones_cross_medida = []

    for query_idx, grupo in topk_cross_df.groupby("query_idx"):
        grupo = grupo.sort_values("rank_cross").copy()
        compatibles = grupo[grupo["medida_coincide"] == True]

        if len(compatibles) > 0:
            elegido = compatibles.iloc[0]
            criterio = "cross_con_medida"
        else:
            elegido = grupo.iloc[0]
            criterio = "cross_sin_medida"

        selecciones_cross_medida.append({
            "query_idx": query_idx,
            "concepto_query": elegido["concepto_query"],
            "concepto_esperado": elegido["concepto_esperado"],
            "concepto_match_cross_medida": elegido["concepto_candidato"],
            "score_cross": elegido["score_cross"],
            "score_emb": elegido["score_emb"],
            "medida_query": elegido["medida_query"],
            "medida_candidato": elegido["medida_candidato"],
            "criterio": criterio
        })

    resultados_cross_medida = pd.DataFrame(selecciones_cross_medida)
    resultados_cross_medida["acierto_top1_cross_medida"] = (
        resultados_cross_medida["concepto_match_cross_medida"].str.strip().str.lower()
        ==
        resultados_cross_medida["concepto_esperado"].str.strip().str.lower()
    )
    accuracy_cross_medida = resultados_cross_medida["acierto_top1_cross_medida"].mean()

    return resultados_cross_medida, accuracy_cross_medida


In [15]:
cross_encoder_candidates = {
    "mmarco_multilingual": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "msmarco_minilm_l6": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "msmarco_minilm_l12": "cross-encoder/ms-marco-MiniLM-L-12-v2"
}

resultados_cross_modelos = {}
resumen_cross = []

for nombre_ce, ce_model_id in cross_encoder_candidates.items():
    print(f"\n=== Evaluando cross-encoder: {nombre_ce} ===")

    cross_encoder = CrossEncoder(ce_model_id)
    topk_cross_df, resultados_cross, accuracy_cross = evaluar_cross_encoder(top5_emb_df, cross_encoder)
    resultados_cross_medida, accuracy_cross_medida = evaluar_cross_encoder_con_medida(topk_cross_df)

    resultados_cross_modelos[nombre_ce] = {
        "model_id": ce_model_id,
        "topk_cross_df": topk_cross_df,
        "resultados_cross": resultados_cross,
        "accuracy_cross": accuracy_cross,
        "resultados_cross_medida": resultados_cross_medida,
        "accuracy_cross_medida": accuracy_cross_medida
    }

    resumen_cross.append({
        "cross_encoder": nombre_ce,
        "model_id": ce_model_id,
        "top1_accuracy_cross": accuracy_cross,
        "top1_accuracy_cross_medida": accuracy_cross_medida
    })

resumen_cross_df = pd.DataFrame(resumen_cross).sort_values(
    by="top1_accuracy_cross_medida", ascending=False
)

display(resumen_cross_df)



=== Evaluando cross-encoder: mmarco_multilingual ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Evaluando cross-encoder: msmarco_minilm_l6 ===


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Evaluando cross-encoder: msmarco_minilm_l12 ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,cross_encoder,model_id,top1_accuracy_cross,top1_accuracy_cross_medida
2,msmarco_minilm_l12,cross-encoder/ms-marco-MiniLM-L-12-v2,0.738095,0.904762
1,msmarco_minilm_l6,cross-encoder/ms-marco-MiniLM-L-6-v2,0.714286,0.880952
0,mmarco_multilingual,cross-encoder/mmarco-mMiniLMv2-L12-H384-v1,0.500000,0.809524


## 7. Resumen final de resultados

La tabla siguiente reúne solo los métodos finalmente considerados relevantes para la comparación principal.
Se excluyen pruebas intermedias descartadas durante la experimentación.

In [16]:
resumen_final_completo = pd.DataFrame([
    {"metodo": "TF-IDF", "top1_accuracy": accuracy_tfidf},
    {"metodo": "Embedding MiniLM multilingual", "top1_accuracy": resultados_modelos["MiniLM_multilingual"]["accuracy_top1"]},
    {"metodo": "Embedding BGE-M3", "top1_accuracy": resultados_modelos["bge_m3"]["accuracy_top1"]},
    {"metodo": "Embedding multilingual-e5-base", "top1_accuracy": resultados_modelos["multilingual_e5_base"]["accuracy_top1"]},
    {"metodo": "Híbrido embedding + medida", "top1_accuracy": accuracy_hibrido},
    {"metodo": "Cross-encoder mMARCO", "top1_accuracy": resultados_cross_modelos["mmarco_multilingual"]["accuracy_cross"]},
    {"metodo": "Cross-encoder mMARCO + medida", "top1_accuracy": resultados_cross_modelos["mmarco_multilingual"]["accuracy_cross_medida"]},
    {"metodo": "Cross-encoder MS MARCO L6", "top1_accuracy": resultados_cross_modelos["msmarco_minilm_l6"]["accuracy_cross"]},
    {"metodo": "Cross-encoder MS MARCO L6 + medida", "top1_accuracy": resultados_cross_modelos["msmarco_minilm_l6"]["accuracy_cross_medida"]},
    {"metodo": "Cross-encoder MS MARCO L12", "top1_accuracy": resultados_cross_modelos["msmarco_minilm_l12"]["accuracy_cross"]},
    {"metodo": "Cross-encoder MS MARCO L12 + medida", "top1_accuracy": resultados_cross_modelos["msmarco_minilm_l12"]["accuracy_cross_medida"]},
]).sort_values(by="top1_accuracy", ascending=False)

display(resumen_final_completo)


,metodo,top1_accuracy
10,Cross-encoder MS MARCO L12 + medida,0.904762
3,Embedding multilingual-e5-base,0.880952
8,Cross-encoder MS MARCO L6 + medida,0.880952
4,Híbrido embedding + medida,0.880952
2,Embedding BGE-M3,0.857143
6,Cross-encoder mMARCO + medida,0.809524
9,Cross-encoder MS MARCO L12,0.738095
7,Cross-encoder MS MARCO L6,0.714286
1,Embedding MiniLM multilingual,0.714286
5,Cross-encoder mMARCO,0.500000


## 8. Mejor pipeline y análisis de errores

Se toma como mejor sistema el que obtiene mayor precisión final.
A continuación se comparan sus predicciones con las del mejor embedding para facilitar el análisis de fallos.

In [17]:
nombre_mejor_ce = "msmarco_minilm_l12"

resultados_cross_best = resultados_cross_modelos[nombre_mejor_ce]["resultados_cross"].copy()
resultados_cross_medida_best = resultados_cross_modelos[nombre_mejor_ce]["resultados_cross_medida"].copy()

comparacion_debug = resultados_emb_best[[
    "concepto_query",
    "concepto_historico_esperado",
    "concepto_match_top1_emb",
    "acierto_top1_exacto_emb"
]].copy()

comparacion_debug = comparacion_debug.rename(columns={
    "concepto_historico_esperado": "concepto_esperado"
})

comparacion_debug["match_cross"] = resultados_cross_best["concepto_match_cross"].values
comparacion_debug["acierto_cross"] = resultados_cross_best["acierto_top1_cross"].values
comparacion_debug["match_cross_medida"] = resultados_cross_medida_best["concepto_match_cross_medida"].values
comparacion_debug["acierto_cross_medida"] = resultados_cross_medida_best["acierto_top1_cross_medida"].values

display(comparacion_debug.head(20))


,concepto_query,concepto_esperado,concepto_match_top1_emb,acierto_top1_exacto_emb,match_cross,acierto_cross,match_cross_medida,acierto_cross_medida
0,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,TAQUETE ARPON 1/2X4 1/2,True,TAQUETE ARPON 1/2X4 1/2,True,TAQUETE ARPON 1/2X4 1/2,True
1,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,PIJA C/FIJ GALV 8X1 1/2,True,PIJA C/FIJ GALV 8X2 1/2,False,PIJA C/FIJ GALV 8X1 1/2,True
2,TAQUETE ARPON 1/4X2 1/4,TAQUETE ARPON 1/4X2 1/4,TAQUETE ARPON 1/4X2 1/4,True,TAQUETE ARPON 1/4X2 1/4,True,TAQUETE ARPON 1/4X2 1/4,True
3,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,True,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,True,BROCA P/CONCRETO 1X12 (SDS-PLUS) BYRNE,True
4,"TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV","TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV","TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV",True,"TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV",True,"TOR C/TRUSS P/BROCA 8X1 1/2"" DR XP GALV",True
5,PIJA C/HEX P/BROCA S/A 12X1 1/2,PIJA C/HEX P/BROCA S/A 12X1 1/2,PIJA C/HEX P/BROCA S/A 12X1 1/2,True,PIJA C/HEX P/BROCA S/A 12X2 1/2,False,PIJA C/HEX P/BROCA S/A 12X1 1/2,True
6,NUDO INOX 3/8-16,NUDO INOX 3/8-16,NUDO INOX 3/8-16,True,NUDO INOX 3/8-16,True,NUDO INOX 3/8-16,True
7,ARANDELA PLANA GALV 1/4,ARANDELA PLANA GALV 1/4,ARANDELA PLANA GALV 1/4,True,ARANDELA PLANA GALV 1/4,True,ARANDELA PLANA GALV 1/4,True
8,"Tubería PVC C80 liso 12""","Tubería PVC Ced-80 EXT LISO de 12""","Tubería PVC Ced-80 EXT LISO de 12""",True,"Tubería PVC Ced-80 EXT LISO de 12""",True,"Tubería PVC Ced-80 EXT LISO de 12""",True
9,"Tubería PVC Ced-80 EXT LISO 1 1/4""","Tubería PVC Ced-80 EXT LISO de 1 1/4""","Tubería PVC Ced-80 EXT LISO de 1 1/4""",True,"Tubería PVC Ced-80 EXT LISO de 1 1/4""",True,"Tubería PVC Ced-80 EXT LISO de 1 1/4""",True
